In [1]:
%pip install torch

import torch
import torch.nn as nn
from torch.nn import functional as F


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
torch.manual_seed(2077)

B,T,C = 4, 8, 16 # batch, time, channels
heads_size = 8
num_heads = C // heads_size
x = torch.randn(B,T,C)

W_Q = nn.Linear(C, C, bias=False)
W_K = nn.Linear(C, C, bias=False)
W_V = nn.Linear(C, C, bias=False)
W_O = nn.Linear(C, C, bias=False )

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q,  K.transpose(-2, -1)) / (d_k ** 0.5)
    
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

# print(x.shape)
# output, attention_weights = scaled_dot_product_attention(W_Q(x), W_K(x), W_V(x))
# print(output.shape)

def multi_head_attention(x, W_Q, W_K, W_V, W_O, num_heads):
    B, T, C = x.shape
    head_size = C // num_heads
    
    # (B, T, C)
    Q = W_Q(x)
    K = W_K(x)
    V = W_V(x)
    
    # (B, T, C) -> (B, T, num_heads, head_size) -> (B, num_heads, T, head_size)
    Q = Q.view(B, T, num_heads, head_size).transpose(1, 2)
    K = K.view(B, T, num_heads, head_size).transpose(1, 2)
    V = V.view(B, T, num_heads, head_size).transpose(1, 2)
    
    attended_values, attention_weights = scaled_dot_product_attention(Q, K, V)
    
    output = attended_values.transpose(1, 2).contiguous().view(B, T, C)
    
    output = W_O(output)
    
    return output, attention_weights

output, attn_weights = multi_head_attention(x, W_Q, W_K, W_V, W_O, num_heads)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")



Input shape: torch.Size([4, 8, 16])
Output shape: torch.Size([4, 8, 16])
Attention weights shape: torch.Size([4, 2, 8, 8])
